#### Import Python packages 

In [1]:
# Import Python packages 
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import json
import csv

#### Creating list of filepaths to process original event csv data files

In [19]:
# checking current working directory
print(os.getcwd())

# Get current folder and subfolder event data
filepath = os.getcwd() + '/event_data'

# Create a for loop to create a list of files and collect each filepath
for root, dirs, files in os.walk(filepath):
    
# join the file path and roots with the subdirectories using glob
    file_path_list = glob.glob(os.path.join(root,'*'))
    #print(file_path_list)

/workspace/home


#### Processing the files to create the data file csv that will be used for Apache Casssandra tables

In [3]:
# initiating an empty list of rows that will be generated from each file
full_data_rows_list = [] 
    
# for every filepath in the file path list 
for f in file_path_list:

# reading csv file 
    with open(f, 'r', encoding = 'utf8', newline='') as csvfile: 
        # creating a csv reader object 
        csvreader = csv.reader(csvfile) 
        next(csvreader)
        
 # extracting each data row one by one and append it        
        for line in csvreader:
            full_data_rows_list.append(line) 
            
print(len(full_data_rows_list))

# creating a smaller event data csv file called event_datafile_full csv that will be used to insert data into the \
# Apache Cassandra tables
csv.register_dialect('myDialect', quoting=csv.QUOTE_ALL, skipinitialspace=True)

with open('event_datafile_new.csv', 'w', encoding = 'utf8', newline='') as f:
    writer = csv.writer(f, dialect='myDialect')
    writer.writerow(['artist','firstName','gender','itemInSession','lastName','length',\
                'level','location','sessionId','song','userId'])
    for row in full_data_rows_list:
        if (row[0] == ''):
            continue
        writer.writerow((row[0], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[12], row[13], row[16]))


8056


In [4]:
with open('event_datafile_new.csv', 'r', encoding = 'utf8') as f:
    print(sum(1 for line in f))

6821


#### Creating a Cluster

In [5]:
from cassandra.cluster import Cluster
cluster = Cluster()

# To establish connection and begin executing queries, need a session
session = cluster.connect()

#### Create Keyspace

In [6]:
query = """
CREATE KEYSPACE IF NOT EXISTS sp
WITH REPLICATION =
{'class': 'SimpleStrategy', 'replication_factor': 1}
"""
session.execute(query)

#### Set Keyspace

In [7]:
session.set_keyspace('sp')

### Query 1
Return the **artist name, song title, and song length** that were played in the music app history for:

- sessionId = 338  
- itemInSession = 4

This query identifies the specific song played at a particular position within a user session.

In [8]:
## Query 1:  Give me the artist, song title and song's length in the music app history that was heard during \
## sessionId = 338, and itemInSession = 4
query2 = """ CREATE TABLE IF NOT EXISTS song_session (
sessionId int,
itemInSession int,
artist text,
song text,
length float,
PRIMARY KEY (sessionId, itemInSession) 
)
""" 
##I chose sessionId as the partition key because the query filters by sessionId, and itemInSession as the clustering column to identify the specific song within that session.
session.execute(query2)

In [9]:
# We have provided part of the code to set up the CSV file. Please complete the Apache Cassandra code below#
file = 'event_datafile_new.csv'

with open(file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader) # skip header
    for line in csvreader:
## Insert data into 'song_session' table from the prepared CSV
        query = "INSERT INTO song_session (sessionId, itemInSession, artist, song, length)"
        query = query + " VALUES (%s,%s,%s,%s,%s)"
        session.execute(query, (int(line[8]), int(line[3]), line[0], line[9], float(line[5])))

#### Do a SELECT to verify that the data have been inserted into each table

In [10]:
query1 = "SELECT artist, song, length FROM song_session WHERE sessionId=338 AND itemInSession=4"
rows = session.execute(query1)
df1 = pd.DataFrame(list(rows))
print(df1)

      artist                             song      length
0  Faithless  Music Matters (Mark Knight Dub)  495.307312


### Query 2
Return the following information:

- Artist name  
- Song title (sorted by itemInSession)  
- User first name and last name  

For the following conditions:

- userId = 10  
- sessionId = 182

This query shows the songs listened to by a specific user during a specific session, ordered by the sequence in which they were played.

In [11]:
## Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name)\
## for userid = 10, sessionid = 182
query4 = """ CREATE TABLE IF NOT EXISTS songs_user_session (
userId int,
sessionId int,
itemInSession int,
artist text,
song text,
firstName text,
lastName text,
PRIMARY KEY ((userId, sessionId), itemInSession)
);
"""
#ً#I chose (userId, sessionId) as the partition key because the query filters by both values, and itemInSession as the clustering column to keep the songs ordered within the session.
session.execute(query4)
                    

In [12]:
file = 'event_datafile_new.csv'
with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)
    
    for line in csvreader:
        query = "INSERT INTO songs_user_session (userId, sessionId, itemInSession, artist, song, firstName, lastName)"
        query = query + " VALUES (%s,%s,%s,%s,%s,%s,%s)"
        session.execute(query, (int(line[10]), int(line[8]), int(line[3]), line[0], line[9], line[1], line[4]))

In [13]:
query2 = "SELECT artist, song, firstName, lastName FROM songs_user_session WHERE userId=10 AND sessionId=182"
rows = session.execute(query2)
df2 = pd.DataFrame(list(rows))
print(df2)

              artist                                               song  \
0   Down To The Bone                                 Keep On Keepin' On   
1       Three Drives                                        Greece 2000   
2  Sebastien Tellier                                          Kilometer   
3      Lonnie Gordon  Catch You Baby (Steve Pitron & Max Sanna Radio...   

  firstname lastname  
0    Sylvie     Cruz  
1    Sylvie     Cruz  
2    Sylvie     Cruz  
3    Sylvie     Cruz  


### Query 3
Return **every user’s first and last name** who listened to the song:

- 'All Hands Against His Own'

This query identifies all users who listened to a particular song in the music app history.

In [14]:
## Query 3: Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'

query5 = """ CREATE TABLE IF NOT EXISTS users_by_song (
song text,
userId int,
firstName text,
lastName text,
PRIMARY KEY (song, userId)
);
"""
##I chose song as the partition key because the query searches for users who listened to a specific song, and userId as the clustering column to identify each user.
session.execute(query5)              

In [15]:
file = 'event_datafile_new.csv'
with open(file, encoding='utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)
    
    for line in csvreader:
        query = "INSERT INTO users_by_song (song, userId, firstName, lastName)"
        query = query + " VALUES (%s,%s,%s,%s)"
        session.execute(query, (line[9], int(line[10]), line[1], line[4]))

In [16]:
query6 = """ SELECT firstName, lastName
FROM users_by_song
WHERE song = 'All Hands Against His Own';
"""
rows = session.execute(query6)
rows_list = list(rows)
df = pd.DataFrame(rows_list)
print(df)

    firstname lastname
0  Jacqueline    Lynch
1       Tegan   Levine
2        Sara  Johnson


### Drop the tables before closing out the sessions

In [17]:
## Drop the table before closing out the sessions

session.execute("DROP TABLE IF EXISTS song_session")
session.execute("DROP TABLE IF EXISTS songs_user_session")
session.execute("DROP TABLE IF EXISTS users_by_song")

### Close the session and cluster connection¶

In [18]:
## Close Cassandra session and cluster connection to free resources
session.shutdown()
cluster.shutdown()